In [ ]:
# 1. IMPORT LIBRARIES & SETUP
from google.colab import files
import pandas as pd
import numpy as np
import io
from sklearn.preprocessing import LabelEncoder

print("Please upload your TWO season CSV files now:")
uploaded = files.upload()
filenames = list(uploaded.keys())

if len(filenames) < 2:
    print("⚠️ Please run the cell again and upload exactly two CSV files.")
else:
    # 2. LOAD & MERGE DATA
    df1 = pd.read_csv(io.BytesIO(uploaded[filenames[0]]))
    df2 = pd.read_csv(io.BytesIO(uploaded[filenames[1]]))

    df1['Season'] = 1
    df2['Season'] = 2
    df = pd.concat([df1, df2], ignore_index=True)

    print(f"✅ Loaded raw data: {df.shape[0]} matches.")

    # 3. CLEANING & INTEGRITY
    # Standardize Dates
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['Date', 'FTHG', 'FTAG', 'FTR']).copy()
    df = df.sort_values(['Date', 'Time'] if 'Time' in df.columns else ['Date']).reset_index(drop=True)

    # Strip string columns
    string_cols = ['HomeTeam', 'AwayTeam', 'FTR', 'HTR', 'Div']
    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    # Enforce integer types for goals
    goal_cols = ['FTHG', 'FTAG', 'HTHG', 'HTAG']
    for col in goal_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(int)

    # Logic Check 1: HT goals cannot exceed FT goals
    valid_goals = (df['FTHG'] >= df['HTHG']) & (df['FTAG'] >= df['HTAG'])
    # Logic Check 2: Result matches goal difference
    valid_ftr = (
        ((df['FTHG'] > df['FTAG']) & (df['FTR'] == 'H')) |
        ((df['FTHG'] < df['FTAG']) & (df['FTR'] == 'A')) |
        ((df['FTHG'] == df['FTAG']) & (df['FTR'] == 'D'))
    )
    df = df[valid_goals & valid_ftr].copy()
    print(f"🧹 Data cleaned. Kept {df.shape[0]} logically sound matches.")

    # 4. MATCH-LEVEL FEATURE ENGINEERING
    df['TotalGoals'] = df['FTHG'] + df['FTAG']
    df['GoalDiff'] = df['FTHG'] - df['FTAG'] # Positive = Home Dominance, Negative = Away Dominance

    # Points awarded
    df['HomePoints'] = np.select([df['FTR'] == 'H', df['FTR'] == 'D', df['FTR'] == 'A'], [3, 1, 0])
    df['AwayPoints'] = np.select([df['FTR'] == 'A', df['FTR'] == 'D', df['FTR'] == 'H'], [3, 1, 0])

    # 5. ML-SAFE ROLLING FEATURES (Avoiding Target Leakage)
    # We must calculate form based strictly on past games.
    # To do this cleanly, we create a temporary melted dataframe of all team performances.
    team_stats = []
    for idx, row in df.iterrows():
        # Home perspective
        team_stats.append({
            'MatchID': idx, 'Team': row['HomeTeam'], 'Date': row['Date'],
            'Points': row['HomePoints'], 'Scored': row['FTHG'], 'Conceded': row['FTAG']
        })
        # Away perspective
        team_stats.append({
            'MatchID': idx, 'Team': row['AwayTeam'], 'Date': row['Date'],
            'Points': row['AwayPoints'], 'Scored': row['FTAG'], 'Conceded': row['FTHG']
        })

    history_df = pd.DataFrame(team_stats).sort_values(['Team', 'Date'])

    # Calculate rolling 5-game stats. shift(1) ensures the current game's stats are NOT included.
    history_df['PreMatch_Form_Pts'] = history_df.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    history_df['PreMatch_AvgScored'] = history_df.groupby('Team')['Scored'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    history_df['PreMatch_AvgConceded'] = history_df.groupby('Team')['Conceded'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    history_df.fillna(0, inplace=True) # Fill NaNs for the very first match of the season

    # Merge rolling features back into the main match dataframe
    home_history = history_df.add_prefix('HomeTeam_').rename(columns={'HomeTeam_MatchID': 'MatchID', 'HomeTeam_Team': 'HomeTeam'})
    away_history = history_df.add_prefix('AwayTeam_').rename(columns={'AwayTeam_MatchID': 'MatchID', 'AwayTeam_Team': 'AwayTeam'})

    df['MatchID'] = df.index
    df = df.merge(home_history[['MatchID', 'HomeTeam_PreMatch_Form_Pts', 'HomeTeam_PreMatch_AvgScored', 'HomeTeam_PreMatch_AvgConceded']], on='MatchID', how='left')
    df = df.merge(away_history[['MatchID', 'AwayTeam_PreMatch_Form_Pts', 'AwayTeam_PreMatch_AvgScored', 'AwayTeam_PreMatch_AvgConceded']], on='MatchID', how='left')
    df.drop(columns=['MatchID'], inplace=True)

    # 6. DATA ENCODING
    # Ordinal encoding for outcomes (useful for correlations and regression)
    result_map = {'A': 0, 'D': 1, 'H': 2}
    df['FTR_Encoded'] = df['FTR'].map(result_map)
    df['HTR_Encoded'] = df['HTR'].map(result_map)

    # Label Encoding for categorical variables (Teams and Divisions)
    le = LabelEncoder()
    # Fit on all unique teams (Home and Away) to ensure consistent encoding
    all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()
    le.fit(all_teams)

    df['HomeTeam_Encoded'] = le.transform(df['HomeTeam'])
    df['AwayTeam_Encoded'] = le.transform(df['AwayTeam'])
    df['Div_Encoded'] = LabelEncoder().fit_transform(df['Div'])

    # Optional: One-Hot Encoding for Match Results (Creates 3 separate binary columns: FTR_H, FTR_D, FTR_A)
    # df = pd.get_dummies(df, columns=['FTR'], prefix='FTR', drop_first=False)

    print(f"⚙️ Engineering complete. DataFrame now has {df.shape[1]} columns.")
    display(df.head())

Please upload your TWO season CSV files now:


Saving 2023-24_E0.csv to 2023-24_E0.csv
Saving 2024-25_E0.csv to 2024-25_E0.csv
✅ Loaded raw data: 760 matches.
🧹 Data cleaned. Kept 760 logically sound matches.
⚙️ Engineering complete. DataFrame now has 148 columns.


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,HomeTeam_PreMatch_AvgScored,HomeTeam_PreMatch_AvgConceded,AwayTeam_PreMatch_Form_Pts,AwayTeam_PreMatch_AvgScored,AwayTeam_PreMatch_AvgConceded,FTR_Encoded,HTR_Encoded,HomeTeam_Encoded,AwayTeam_Encoded,Div_Encoded
0,E0,2023-08-11,20:00,Burnley,Man City,0,3,A,0,2,...,0.0,0.0,0.0,0.0,0.0,0,0,5,14,0
1,E0,2023-08-11,20:00,Burnley,Man City,0,3,A,0,2,...,0.0,0.0,0.0,0.0,0.0,0,0,5,14,0
2,E0,2023-08-11,20:00,Burnley,Man City,0,3,A,0,2,...,0.0,0.0,0.0,0.0,0.0,0,0,5,14,0
3,E0,2023-08-11,20:00,Burnley,Man City,0,3,A,0,2,...,0.0,0.0,0.0,0.0,0.0,0,0,5,14,0
4,E0,2023-08-12,12:30,Arsenal,Nott'm Forest,2,1,H,2,0,...,0.0,0.0,0.0,0.0,0.0,2,2,0,17,0


In [ ]:
# 1. Define feature categories for clarity
context_cols = [
    'Date', 'HomeTeam', 'AwayTeam'  # Kept for Plotly hover-text and time-series plotting
]

target_cols = [
    'FTR', 'FTR_Encoded',           # The categorical and numeric match outcome
    'FTHG', 'FTAG',                 # Actual goals scored
    'TotalGoals', 'GoalDiff'        # Engineered outcomes for regression/distribution plots
]

predictor_cols = [
    'HomeTeam_Encoded', 'AwayTeam_Encoded', # ML-readable team identifiers
    'HomeTeam_PreMatch_Form_Pts',           # 5-game rolling points (Home)
    'HomeTeam_PreMatch_AvgScored',          # 5-game rolling attack (Home)
    'HomeTeam_PreMatch_AvgConceded',        # 5-game rolling defense (Home)
    'AwayTeam_PreMatch_Form_Pts',           # 5-game rolling points (Away)
    'AwayTeam_PreMatch_AvgScored',          # 5-game rolling attack (Away)
    'AwayTeam_PreMatch_AvgConceded'         # 5-game rolling defense (Away)
]

# 2. Create the lean analysis dataframe
df_analysis = df[context_cols + target_cols + predictor_cols].copy()

# Display the exact columns kept and check for any remaining missing values
print(f"Created df_analysis with {df_analysis.shape[1]} columns.")
df_analysis.info()

Created df_analysis with 17 columns.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3040 entries, 0 to 3039
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           3040 non-null   datetime64[ns]
 1   HomeTeam                       3040 non-null   object        
 2   AwayTeam                       3040 non-null   object        
 3   FTR                            3040 non-null   object        
 4   FTR_Encoded                    3040 non-null   int64         
 5   FTHG                           3040 non-null   int64         
 6   FTAG                           3040 non-null   int64         
 7   TotalGoals                     3040 non-null   int64         
 8   GoalDiff                       3040 non-null   int64         
 9   HomeTeam_Encoded               3040 non-null   int64         
 10  AwayTeam_Encoded               3040 non-null   

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# Assuming 'df' is your cleaned dataframe with the original columns
# Recalculate Total Goals just in case it's missing
df['TotalGoals'] = df['FTHG'] + df['FTAG']

# --- PLOT 1: RELATIONSHIP (Half-Time vs Full-Time Home Goals) ---
# We add "jitter" (tiny random noise) so the dots don't completely overlap on the integers
df['HTHG_Jitter'] = df['HTHG'] + np.random.normal(0, 0.1, size=len(df))
df['FTHG_Jitter'] = df['FTHG'] + np.random.normal(0, 0.1, size=len(df))

fig1 = px.scatter(df, x='HTHG_Jitter', y='FTHG_Jitter', opacity=0.4,
                  title="1. Half-Time vs. Full-Time Home Goals",
                  labels={'HTHG_Jitter': 'Half-Time Home Goals', 'FTHG_Jitter': 'Full-Time Home Goals'},
                  color_discrete_sequence=['#1f77b4'])
fig1.update_layout(height=400)
fig1.show()

# --- PLOT 2: CORRELATION HEATMAP (All Goal Metrics) ---
# A simple grid showing how strongly the goal metrics relate to each other
goal_cols = ['HTHG', 'HTAG', 'FTHG', 'FTAG', 'TotalGoals']
corr_matrix = df[goal_cols].corr()

fig2 = px.imshow(corr_matrix, text_auto=".2f", aspect="auto",
                 title="2. Goal Correlation Heatmap",
                 color_continuous_scale='Blues')
fig2.update_layout(height=400)
fig2.show()

# --- PLOT 3: COMPOSITION (Overall Match Results) ---
# The simplest breakdown of Home Wins, Draws, and Away Wins
fig3 = px.pie(df, names='FTR', title="3. Full-Time Results Breakdown",
              color='FTR', color_discrete_map={'H':'#2ca02c', 'D':'#7f7f7f', 'A':'#d62728'},
              hole=0.4) # Makes it a modern donut chart
fig3.update_traces(textinfo='percent+label')
fig3.update_layout(height=400)
fig3.show()

# --- PLOT 4: DISTRIBUTION (Home vs Away Goals) ---
# Overlaid histograms to show exactly how much of an advantage playing at home gives
fig4 = px.histogram(df, x=['FTHG', 'FTAG'], barmode='group',
                    title="4. Goals Scored: Home vs. Away Teams",
                    labels={'value': 'Goals Scored', 'variable': 'Team'},
                    color_discrete_map={'FTHG': '#1f77b4', 'FTAG': '#ff7f0e'})
fig4.update_layout(xaxis_title="Number of Goals in a Match", yaxis_title="Count of Matches", height=400)
fig4.show()

# --- PLOT 5: TRANSITION (Half-Time Result to Full-Time Result) ---
# How often does a Half-Time Draw turn into a Full-Time Win?
fig5 = px.histogram(df, x='HTR', color='FTR', barmode='group',
                    title="5. How Half-Time Results Convert to Full-Time Results",
                    labels={'HTR': 'Half-Time Result (H/D/A)'},
                    category_orders={'HTR': ['H', 'D', 'A'], 'FTR': ['H', 'D', 'A']},
                    color_discrete_map={'H':'#2ca02c', 'D':'#7f7f7f', 'A':'#d62728'})
fig5.update_layout(yaxis_title="Count of Matches", height=400)
fig5.show()

# --- PLOT 6: FREQUENCY (Total Goals per Match) ---
# A simple count of how common 0-0, 1-0, 2-1, etc., matches are
fig6 = px.histogram(df, x='TotalGoals', text_auto=True,
                    title="6. Frequency of Total Goals per Match",
                    labels={'TotalGoals': 'Total Goals (Home + Away)'})
fig6.update_layout(yaxis_title="Count of Matches", xaxis=dict(tickmode='linear', tick0=0, dtick=1), height=400)
fig6.show()

In [ ]:
import plotly.express as px
import pandas as pd

# Ensure Date is datetime to extract months
df['Date'] = pd.to_datetime(df['Date'])
df['MonthName'] = df['Date'].dt.month_name()
df['GoalDiff'] = df['FTHG'] - df['FTAG']

# --- PLOT 7: THE SCORELINE MATRIX (Exact Match Results) ---
# Creates a grid showing exactly how many matches ended 1-0, 0-0, 2-1, etc.
score_matrix = pd.crosstab(df['FTHG'], df['FTAG'])
fig7 = px.imshow(score_matrix, text_auto=True, origin='lower',
                 title="7. Exact Scoreline Frequency",
                 labels=dict(x="Away Team Goals", y="Home Team Goals", color="Matches"),
                 color_continuous_scale='YlGnBu')
fig7.update_layout(height=500, xaxis=dict(tickmode='linear', dtick=1), yaxis=dict(tickmode='linear', dtick=1))
fig7.show()

# --- PLOT 8: TOP 10 HOME SCORING TEAMS (Horizontal Bar Chart) ---
# Sums up all goals scored at home by each team and ranks the top 10
top_home_scorers = df.groupby('HomeTeam')['FTHG'].sum().nlargest(10).reset_index()
top_home_scorers = top_home_scorers.sort_values('FTHG', ascending=True) # Sort for Plotly horizontal

fig8 = px.bar(top_home_scorers, x='FTHG', y='HomeTeam', orientation='h',
              title="8. Top 10 Most Lethal Home Teams (Total Home Goals)",
              labels={'FTHG': 'Total Home Goals Scored', 'HomeTeam': 'Team'},
              color='FTHG', color_continuous_scale='Reds')
fig8.update_layout(height=400)
fig8.show()

# --- PLOT 9: GOALS BY RESULT (Box Plot) ---
# Shows if draws are usually boring (0-0) and if wins are high-scoring
fig9 = px.box(df, x='FTR', y='TotalGoals', color='FTR',
              title="9. Total Goals Spread by Match Result",
              labels={'FTR': 'Full Time Result', 'TotalGoals': 'Total Goals in Match'},
              color_discrete_map={'H':'#2ca02c', 'D':'#7f7f7f', 'A':'#d62728'},
              category_orders={'FTR': ['H', 'D', 'A']})
fig9.update_layout(height=400)
fig9.show()

# --- PLOT 10: GOAL DIFFERENCE DISTRIBUTION (Histogram) ---
# Positive numbers mean Home blowouts, negative mean Away blowouts
fig10 = px.histogram(df, x='GoalDiff', text_auto=True,
                     title="10. Goal Difference Spread (Home vs Away Dominance)",
                     labels={'GoalDiff': 'Goal Difference (Home Goals - Away Goals)'},
                     color_discrete_sequence=['#9467bd'])
fig10.update_layout(xaxis=dict(tickmode='linear', dtick=1), yaxis_title="Count of Matches", height=400)
fig10.show()

# --- PLOT 11: SEASONALITY (Matches by Month) ---
# Shows the busy and quiet periods of the football calendar
month_order = ['August', 'September', 'October', 'November', 'December',
               'January', 'February', 'March', 'April', 'May', 'June', 'July']
fig11 = px.histogram(df, x='MonthName', category_orders={'MonthName': month_order},
                     title="11. Football Calendar: Matches Played per Month",
                     labels={'MonthName': 'Month'},
                     color_discrete_sequence=['#17becf'])
fig11.update_layout(yaxis_title="Total Matches Played", height=400)
fig11.show()

In [ ]:
import plotly.express as px
import pandas as pd

# 1. ENGINEER SECOND-HALF METRICS
df['SHHG'] = df['FTHG'] - df['HTHG']  # Second Half Home Goals
df['SHAG'] = df['FTAG'] - df['HTAG']  # Second Half Away Goals
df['HT_GoalDiff'] = df['HTHG'] - df['HTAG']
df['SH_GoalDiff'] = df['SHHG'] - df['SHAG']

# --- PLOT A: 1st Half Goals vs 2nd Half Goals (Density Heatmap) ---
# A scatter plot is too messy for small integers. A 2D density heatmap shows exactly
# where the most common match flows occur.
figA = px.density_heatmap(df, x='HTHG', y='SHHG', text_auto=True,
                          title="A. Home Team Momentum: 1st Half Goals vs 2nd Half Goals",
                          labels={'HTHG': '1st Half Goals Scored', 'SHHG': '2nd Half Goals Scored'},
                          color_continuous_scale='Blues')
figA.update_layout(xaxis=dict(tickmode='linear', dtick=1), yaxis=dict(tickmode='linear', dtick=1), height=400)
figA.show()

# --- PLOT B: Goal Difference Shift (Scatter with Trendline) ---
# Does dominating the first half mean you will dominate the second half?
# Jitter is added for visibility.
import numpy as np
df['HT_GD_Jitter'] = df['HT_GoalDiff'] + np.random.normal(0, 0.15, size=len(df))
df['SH_GD_Jitter'] = df['SH_GoalDiff'] + np.random.normal(0, 0.15, size=len(df))

figB = px.scatter(df, x='HT_GD_Jitter', y='SH_GD_Jitter', opacity=0.3, color='FTR',
                  title="B. The Momentum Shift: HT Goal Difference vs. 2nd Half Goal Difference",
                  labels={'HT_GD_Jitter': 'Half-Time Goal Diff (+ is Home Lead)',
                          'SH_GD_Jitter': 'Second-Half Goal Diff (+ is Home Dominance in 2nd Half)'},
                  color_discrete_map={'H':'#2ca02c', 'D':'#7f7f7f', 'A':'#d62728'})
figB.add_hline(y=0, line_dash="dash", line_color="black")
figB.add_vline(x=0, line_dash="dash", line_color="black")
figB.update_layout(height=500)
figB.show()

# --- PLOT C: When are goals scored? (Grouped Bar Chart) ---
# A simple comparison of league-wide scoring across halves
halves_data = pd.DataFrame({
    'Half': ['First Half', 'Second Half', 'First Half', 'Second Half'],
    'Team': ['Home', 'Home', 'Away', 'Away'],
    'Total Goals': [df['HTHG'].sum(), df['SHHG'].sum(), df['HTAG'].sum(), df['SHAG'].sum()]
})

figC = px.bar(halves_data, x='Half', y='Total Goals', color='Team', barmode='group',
              title="C. League Total Goals: First Half vs. Second Half",
              color_discrete_map={'Home': '#1f77b4', 'Away': '#ff7f0e'})
figC.update_layout(height=400)
figC.show()